In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

def processar_dre_bcb(caminho_arquivo):
    """
    Lê e padroniza um único arquivo CSV de DRE do Banco Central.
    """
    try:
        df_headers = pd.read_csv(caminho_arquivo, sep=';', encoding='utf-8', nrows=2, header=None)
        
        colunas_finais = []
        for i in range(df_headers.shape[1]):
            pass
        
        df = pd.read_csv(
            caminho_arquivo, 
            sep=';', 
            encoding='utf-8', 
            skiprows=[1, 2], 
            decimal=',',     
            thousands='.',   
            na_values=['', '-', 'NI'] 
        )
        
        if df.columns[-1].startswith('Unnamed'):
            df = df.drop(columns=[df.columns[-1]])
            
        df = df.dropna(subset=['Instituição', 'Data'])
        
        df['Data'] = pd.to_datetime(df['Data'], format='%m/%Y', errors='coerce')
        
        for col in df.columns:
            if df[col].dtype == 'object' and df[col].str.contains('%').any():
                df[col] = df[col].str.replace('%', '').str.replace(',', '.').astype(float) / 100
                
        return df

    except Exception as e:
        print(f"Erro ao processar {caminho_arquivo.name}: {e}")
        return None

def consolidar_dres(diretorio_raiz):
    """
    Varre pastas e subpastas buscando CSVs e os consolida em um único DataFrame.
    """
    pasta_raiz = Path(diretorio_raiz)
    arquivos_csv = list(pasta_raiz.rglob('*.csv'))
    print(f"Encontrados {len(arquivos_csv)} arquivos para processar.")
    
    lista_dfs = []
    
    for arquivo in arquivos_csv:
        print(f"Processando: {arquivo.name}...")
        df_processado = processar_dre_bcb(arquivo)
        
        if df_processado is not None and not df_processado.empty:
            lista_dfs.append(df_processado)
            
    # Concatena todos os arquivos empilhando-os
    if lista_dfs:
        df_final = pd.concat(lista_dfs, ignore_index=True)
        # Ordena o painel por Instituição e depois por Data no tempo
        df_final = df_final.sort_values(by=['Instituição', 'Data']).reset_index(drop=True)
        print("Consolidação concluída com sucesso!")
        return df_final
    else:
        print("Nenhum dado pôde ser consolidado.")
        return pd.DataFrame()


CAMINHO_DA_PASTA = './home/inovar-para-pessoas-negras/Área de trabalho/gustavo/bank-distress-foretoken' 

df_painel_bancos = consolidar_dres(CAMINHO_DA_PASTA)

# Exibe as primeiras linhas e a estrutura para conferência
print(df_painel_bancos.head())
print(df_painel_bancos.info())

SyntaxError: invalid syntax (2306704981.py, line 9)

In [ ]:
import torch
import pandas as pd
import glob
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data

# ---------------------------------------------------------
# 1. Definição da Arquitetura do Modelo (GNN)
# ---------------------------------------------------------
class BankBankruptcyGCN(torch.nn.Module):
    def __init__(self, num_node_features, hidden_channels):
        super(BankBankruptcyGCN, self).__init__()
        # Primeira camada convolucional em grafo
        self.conv1 = GCNConv(num_node_features, hidden_channels)
        # Segunda camada convolucional
        self.conv2 = GCNConv(hidden_channels, hidden_channels // 2)
        # Camada linear de saída para classificação binária (0 = Saudável, 1 = Falência)
        self.out = torch.nn.Linear(hidden_channels // 2, 2)

    def forward(self, x, edge_index):
        # x: Matriz de atributos dos nós (Bancos x Variáveis do COSIF)
        # edge_index: Estrutura do grafo (Similaridade entre bancos)
        
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training) # Previne overfitting
        
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        
        x = self.out(x)
        return x # Retorna os logits brutos



motherfolder = r"/home/inovar-para-pessoas-negras/Área de trabalho/gustavo/bank-distress-foretoken/dres"
minorfolders = [f for f in glob.glob(motherfolder = "/*.csv")]

summarized_csv = pd.Dataframe(columns = [])

# leitura dos arquivos e obtenção das informações
summary_table = pd.DataFrame(columns = ['Instituição';'Código';'TCB';'SR';'TD';'TC';'Cidade';'UF';'Data';'Resultado de Intermediação Financeira'])

for file in minorfolders:
    try:
        f = pd.read_csv(file, sep = ";") 
        info = [file.split(';;;;;;;;;;;;;;')[len(file.split(';;;;;;;;;;;;;;'))-1], 
                f.shape[0], 
                f.shape[1]-1, 
                f['Data'].min(),
                f['Data'].max(), 
                f.isna().sum().sum()] 
        
        new_info = pd.DataFrame([info], columns = ['Instituição';'Código';'TCB';'SR';'TD';'TC';'Cidade';'UF';'Data';'Resultado de Intermediação Financeira'])
        summary_table = pd.concat([summary_table, new_info], ignore_index = True)
    except:
        pass

#num_bancos = 100
#num_features = 15

# Tensor de features dos nós (X)
#x = torch.rand((num_bancos, num_features), dtype=torch.float)

# Tensor de arestas (Ex: matriz 2xN definindo conexões de similaridade)
# Aqui, o banco 0 está conectado ao 1, o 1 ao 2, etc.
edge_index = torch.tensor([[0, 1, 1, 2, 50, 60],
                           [1, 0, 2, 1, 60, 50]], dtype=torch.long)

# Rótulos das classes (Y): 0 para saudável, 1 para falência
# Desbalanceamento severo: apenas 5 bancos faliram neste cenário
y = torch.zeros(num_bancos, dtype=torch.long)
y[:5] = 1 

# Empacotando no formato do PyTorch Geometric
data = Data(x=x, edge_index=edge_index, y=y)

# ---------------------------------------------------------
# 3. Configuração e Loop de Treinamento
# ---------------------------------------------------------
model = BankBankruptcyGCN(num_node_features=num_features, hidden_channels=32)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

# Como falência é raro, aplicamos pesos na função de perda (Weighted Cross Entropy)
# Dá muito mais peso à classe 1 (Falência) do que à classe 0 (Saudável)
pesos_classes = torch.tensor([1.0, 15.0]) 
criterion = torch.nn.CrossEntropyLoss(weight=pesos_classes)

def train():
    model.train()
    optimizer.zero_grad()  
    out = model(data.x, data.edge_index)  
    loss = criterion(out, data.y)  
    loss.backward()  
    optimizer.step()  
    return loss.item()

# Executando algumas épocas
for epoch in range(1, 101):
    loss = train()
    if epoch % 20 == 0:
        print(f'Época: {epoch:03d}, Loss: {loss:.4f}')